# Stage 01 — Zenodo Source Audit

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design  
**Phase 1:** Source structure, provenance and quality review  
**Preceding checkpoint:** `00_scope_and_source_registry.ipynb`  
**Primary source:** [Zenodo record 19602891](https://zenodo.org/records/19602891)

This notebook extends the Stage 00 format into an executable audit of the rheology archive and image index. It preserves original files and records evidence for the next schema-design stage. Each major section has expandable **Data-science rationale** and **Materials-science context** notes explaining what the checks establish and how their results affect scientific interpretation.

The notebook is self-contained: **Restart Kernel and Run All** recreates its variables and audit tables. It reads workbooks directly from the ZIP, so manual unzipping is unnecessary. Audit exports are regenerated in `data/metadata/zenodo_source_audit/`; rerunning replaces these named audit outputs. The original ZIP and CSV are opened for reading only.

### Scope and evidence levels

- **Observed:** file bytes, workbook cells, headers, counts and measured-value ranges.
- **Source-documented:** statements explicitly linked to the dataset record.
- **Candidate interpretation:** filename correspondences or patterns in changing test variables that require experimental documentation.
- **Unresolved:** sample identity, independent replication, unspecified units or conditions, and image existence without the image archive.

A completed run is an inventory and quality review. It does not by itself validate printability, biological performance, causal effects or machine-learning readiness.

## 1. Source selection and retrieval record

The audit uses `ALG-Ph_HA-Ph_rheology_data.zip` and `image_index.csv`, stored under `data/raw/zenodo_19602891/`. The record page showed version **v0.0.2** when reviewed on **2 October 2026**. Its published MD5 checksums are retained below as a fixed reference. These are review-date metadata, not a live query at every execution.

<details>
<summary><strong>Source download and documentation</strong></summary>

The [source record](https://zenodo.org/records/19602891) provides download links under **Files**. The two selected downloads retain their original names in the local raw-data directory. The browser may append `(1)` to repeat downloads; the notebook accepts exactly one such variant if the canonical name is absent and records the actual name selected.

The image archive and trained models are outside this audit. Image-file existence remains untested. The associated [publication](https://doi.org/10.1080/17452759.2026.2671497) and [authors’ code](https://github.com/KORINZ/generative-ai-bioprinting-framework) are references for subsequent methodological review.

Dataset redistribution permission is recorded as unresolved here: the reviewed webpage text did not expose a clear dataset licence identifier. A licence statement about the article is not automatically a dataset licence. No automated download or raw-data publication occurs in this notebook.

</details>

<details>
<summary><strong>Data-science rationale</strong></summary>

Provenance connects each audit to an identifiable source release and the exact bytes used. A DOI identifies a source record; a checksum helps establish whether a downloaded file matches the published file. MD5 is used only to compare the source’s published checksum; SHA-256 is also recorded for local provenance. A match establishes byte agreement, not scientific accuracy.

Reproducibility requires declared dependencies, deterministic input selection and explicit output locations. A notebook that silently chooses the newest similarly named file can analyse a different input after a restart. Ambiguous duplicate downloads therefore stop execution with a clear message.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Material names provide context, but a polymer formulation is not completely specified by its name. Molecular weight, distribution of chain lengths, degree of substitution, concentration basis, solvent composition and preparation history can affect solution viscosity and network response. Unreported descriptors remain unknown.

The source describes ALG-Ph and HA-Ph inks. The audit preserves the more detailed labels appearing in filenames without assuming that a suffix identifies a distinct synthesis batch or a known substitution level. Documentary evidence is needed before converting labels into chemical features.

</details>

## 2. Execution context and dependencies

The audit requires a compatible Python, NumPy, pandas and openpyxl environment. Jupyter supplies IPython for rich table display; plain Python execution uses a text-display fallback. Actual imported package versions are reported by the running process.

### One-time environment setup for the Windows import error

The reviewed HTML stopped at `import pandas`: NumPy 2.0.2 was loaded from a user-site installation, whereas pandas was loaded from Anaconda and expected the NumPy 1.x binary interface. The remaining audit cells had not executed in that export. This is an environment compatibility failure, not evidence of a problem in the rheology files.

The following commands belong in **Anaconda Prompt**, not in a notebook code cell. They create a separate project environment, register it with Jupyter, and disable user-site package injection for that kernel. The package versions below provide a reproducible Python 3.12 starting configuration. Installation requires an internet connection; routine audit execution does not.

```bat
conda create --name hydrogel-bioink python=3.12 numpy=1.26.4 pandas=2.2.3 openpyxl=3.1.5 ipykernel --yes
conda activate hydrogel-bioink
python -s -m ipykernel install --user --name hydrogel-bioink --display-name "Python (hydrogel-bioink)" --env PYTHONNOUSERSITE 1
```

After setup, the notebook uses **Kernel → Change Kernel → Python (hydrogel-bioink)**, followed by **Restart Kernel and Run All**. Reopening the saved notebook retains this kernel selection. No package installation or environment mutation occurs during Run All. Existing project environments are left in place.

If an environment named `hydrogel-bioink` already exists, its compatibility should be checked before changing it; it need not be recreated on each run. Package installation, missing source files, or an incorrectly selected kernel can still prevent execution. Reproducibility depends on both the notebook and its environment.

Reference: [NumPy troubleshooting — downstream binary incompatibility](https://numpy.org/doc/stable/user/troubleshooting-importerror.html).



<details>
<summary><strong>Data-science rationale</strong></summary>

Parsing depends on software as well as files. Recording versions helps diagnose changes in Excel interpretation, CSV handling and data types. Dependencies are imported once at the beginning; later cells refer only to objects created above them. This supports fresh-kernel execution and makes failures reproducible.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Reproducible parsing preserves the distinction between a reported measurement and its computational representation. A numeric-looking identifier must not become a physical quantity merely because software can convert it to a number. Units and experimental meaning remain attached to the source header.

</details>

In [4]:
from pathlib import Path, PurePosixPath
from collections import Counter
from importlib import metadata
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import platform
import re
import sys
import zipfile
import numbers
# Probe imports in a separate process before loading compiled packages here.
# Inherit this kernel's environment, including PYTHONNOUSERSITE when configured.
import subprocess
probe = subprocess.run(
    [sys.executable, "-c", "import numpy, pandas, openpyxl; print('AUDIT_IMPORTS_OK')"],
    capture_output=True, text=True, errors="replace", timeout=60,
)
if probe.returncode != 0 or "AUDIT_IMPORTS_OK" not in probe.stdout:
    raise RuntimeError(
        "The selected kernel cannot import compatible NumPy, pandas and openpyxl packages. "
        "Complete the one-time Anaconda Prompt setup in Section 2, select "
        "Python (hydrogel-bioink), then Restart Kernel and Run All. "
        "The audit has stopped before reading or writing project data. "
        "The captured dependency diagnostic is available as probe.stderr."
    ) from None
import numpy as np
import pandas as pd
import openpyxl
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not installed (plain Python display fallback available)"

RUN_UTC = datetime.now(timezone.utc).isoformat()
environment = pd.DataFrame([
    {"component": "Python", "version": platform.python_version()},
    *[{"component": p, "version": ({"numpy": np.__version__, "pandas": pd.__version__,
                              "openpyxl": openpyxl.__version__}.get(p) or installed_version(p))}
      for p in ("pandas", "numpy", "openpyxl", "ipython")],
])
display(environment)
print("Audit execution time (UTC):", RUN_UTC)

,component,version
0,Python,3.12.15
1,pandas,2.2.3
2,numpy,1.26.4
3,openpyxl,3.1.5
4,ipython,9.17.1


Audit execution time (UTC): 2026-10-02T07:18:17.591993+00:00


## 3. Project paths and deterministic input selection

The path search checks the current directory and its parents for the designated raw-data folder. A configurable project root is available if Jupyter was launched elsewhere. The displayed paths are relative to the project root.

<details>
<summary><strong>Data-science rationale</strong></summary>

A bounded search avoids confusing this project’s files with another dataset elsewhere on the computer. The canonical filename takes precedence; if only a repeat-download name exists, exactly one candidate is required. Actual filenames are exported in the source manifest. Missing or ambiguous inputs are explicit errors rather than empty datasets.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Selecting the wrong release or a different experiment’s archive can create an apparent material difference that is actually a provenance error. File selection is therefore part of measurement traceability. It does not establish whether the archive’s records represent independent samples.

</details>

In [6]:
# Optional: replace None with Path(r"C:\path\to\hydrogel-bioink") if needed.
PROJECT_ROOT_OVERRIDE = None
RAW_RELATIVE = Path("data/raw/zenodo_19602891")

if PROJECT_ROOT_OVERRIDE is not None:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    project_root = next((p for p in (here, *here.parents)
                         if (p / RAW_RELATIVE).is_dir()), None)
    if project_root is None:
        raise FileNotFoundError(
            "No data/raw/zenodo_19602891 folder found in the current directory or its parents. "
            "Open this notebook from the project notebooks folder, or set PROJECT_ROOT_OVERRIDE.")
raw_dir = project_root / RAW_RELATIVE
output_dir = project_root / "data/metadata/zenodo_source_audit"

def select_input(canonical):
    exact = raw_dir / canonical
    if exact.is_file():
        return exact
    name = Path(canonical)
    pattern = re.compile(re.escape(name.stem) + r"\s*\(\d+\)" + re.escape(name.suffix))
    choices = sorted(p for p in raw_dir.iterdir() if p.is_file() and pattern.fullmatch(p.name))
    if len(choices) != 1:
        raise FileNotFoundError(f"Expected {canonical}, or exactly one numbered variant; found {len(choices)}.")
    return choices[0]

zip_path = select_input("ALG-Ph_HA-Ph_rheology_data.zip")
index_path = select_input("image_index.csv")
for p in (zip_path, index_path):
    print("Selected:", p.relative_to(project_root).as_posix())
print("Audit output folder:", output_dir.relative_to(project_root).as_posix())

Selected: data/raw/zenodo_19602891/ALG-Ph_HA-Ph_rheology_data.zip
Selected: data/raw/zenodo_19602891/image_index.csv
Audit output folder: data/metadata/zenodo_source_audit


## 4. Source checksums and audit issue register

The manifest compares both selected files against the reference checksums. A mismatch is logged and displayed for review; it does not cause the notebook to delete or repair the file.

<details>
<summary><strong>Data-science rationale</strong></summary>

A manifest supplies provenance at file level. An issue register separates observations from decisions: a warning does not automatically exclude data. Each finding records its scope, evidence and next action. Stable issue codes support review across repeated runs; execution time is stored separately from deterministic audit tables.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Instrument data can contain physically unusual values for several reasons, including low torque, phase uncertainty, baseline effects or a protocol outside its valid range. The audit will flag values needing review while retaining the original evidence. Material behaviour should not be inferred from a single warning or discarded because of an arbitrary cleaning rule.

</details>

In [8]:
REFERENCE_MD5 = {
    "ALG-Ph_HA-Ph_rheology_data.zip": "4eaac901afc46212108bfdad8b39b5c2",
    "image_index.csv": "e0fc1a61b6f5d5b2ea87a69a97d4aa58",
}
issues = []
def flag(code, source, detail, action, severity="review", sheet="", column=""):
    issues.append(dict(issue_code=code, severity=severity, source=source,
                       sheet=sheet, column=column, evidence=detail, next_action=action))

def digest(path, algorithm):
    h = hashlib.new(algorithm)
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for canonical, path in zip(REFERENCE_MD5, (zip_path, index_path)):
    actual = digest(path, "md5")
    matched = actual == REFERENCE_MD5[canonical]
    manifest_rows.append(dict(source_record="https://zenodo.org/records/19602891",
        source_version="v0.0.2", metadata_review_date="2026-10-02",
        canonical_filename=canonical, selected_filename=path.name,
        relative_path=path.relative_to(project_root).as_posix(), bytes=path.stat().st_size,
        md5=actual, reference_md5=REFERENCE_MD5[canonical], checksum_match=matched,
        sha256=digest(path, "sha256"), dataset_licence_status="unresolved"))
    if not matched:
        flag("SOURCE_CHECKSUM_MISMATCH", path.name, "Local MD5 differs from the reviewed record.",
             "Verify release, source URL and whether the file was edited before interpreting results.")
source_manifest = pd.DataFrame(manifest_rows)
flag("DATASET_LICENCE_UNRESOLVED", "Zenodo record 19602891",
     "Dataset licence identifier not established in this audit.",
     "Confirm dataset reuse and redistribution terms from authoritative metadata.")
display(source_manifest.drop(columns=["sha256", "md5", "reference_md5"]))

,source_record,source_version,metadata_review_date,canonical_filename,selected_filename,relative_path,bytes,checksum_match,dataset_licence_status
0,https://zenodo.org/records/19602891,v0.0.2,2026-10-02,ALG-Ph_HA-Ph_rheology_data.zip,ALG-Ph_HA-Ph_rheology_data.zip,data/raw/zenodo_19602891/ALG-Ph_HA-Ph_rheology...,1121259,True,unresolved
1,https://zenodo.org/records/19602891,v0.0.2,2026-10-02,image_index.csv,image_index.csv,data/raw/zenodo_19602891/image_index.csv,37937,True,unresolved


## 5. Archive integrity and member inventory

The ZIP directory is inventoried before workbook parsing. File sizes, CRC integrity, exact member paths and content hashes are recorded. No archive member is extracted to disk.

<details>
<summary><strong>Data-science rationale</strong></summary>

Archive member paths are identifiers within the source container. CRC checks detect damaged members, while SHA-256 supports duplicate-content detection. Duplicate names and unsafe paths are audited before any reading. Size bounds prevent an unexpectedly expanded archive from consuming uncontrolled resources. Directory entries are counted separately from data files.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Folder names may suggest measurement families, but are not sufficient experimental metadata. A modulus folder can contain an amplitude sweep, a frequency sweep or another oscillatory protocol. The changing variables and source documentation must support any test-type assignment. Identical workbook bytes may indicate duplicated exports, not additional experimental replication.

</details>

In [10]:
MAX_MEMBER_BYTES = 50 * 1024 * 1024
MAX_ARCHIVE_EXPANDED_BYTES = 250 * 1024 * 1024
with zipfile.ZipFile(zip_path) as archive:
    members = archive.infolist()
    unsafe = [m.filename for m in members if PurePosixPath(m.filename).is_absolute()
              or ".." in PurePosixPath(m.filename).parts or "\\" in m.filename
              or re.match(r"^[A-Za-z]:", m.filename)]
    if unsafe:
        raise ValueError("Archive contains unexpected member paths: " + repr(unsafe[:5]))
    counts = Counter(m.filename for m in members)
    if any(n > 1 for n in counts.values()):
        raise ValueError("Duplicate ZIP member names require review before parsing.")
    if any(m.file_size > MAX_MEMBER_BYTES for m in members) or sum(m.file_size for m in members) > MAX_ARCHIVE_EXPANDED_BYTES:
        raise ValueError("Archive exceeds the declared audit size bounds; review before increasing limits.")
    bad_member = archive.testzip()
    if bad_member is not None:
        raise ValueError("CRC integrity failure: " + bad_member)
    archive_rows = []
    for m in members:
        archive_rows.append(dict(member_path=m.filename, is_directory=m.is_dir(),
            folder=PurePosixPath(m.filename).parent.as_posix(),
            suffix=PurePosixPath(m.filename).suffix.lower(), bytes=m.file_size,
            compressed_bytes=m.compress_size, crc32=f"{m.CRC:08x}",
            sha256="" if m.is_dir() else hashlib.sha256(archive.read(m)).hexdigest()))
archive_inventory = pd.DataFrame(archive_rows)
files = archive_inventory.loc[~archive_inventory.is_directory].copy()
xlsx_members = sorted(files.loc[files.suffix.eq(".xlsx"), "member_path"].tolist())
if not xlsx_members:
    raise ValueError("No XLSX workbooks found in this archive.")
for _, row in files.loc[files.duplicated("sha256", keep=False)].iterrows():
    flag("DUPLICATE_MEMBER_CONTENT", row.member_path, "SHA-256 matches another archived file.",
         "Determine whether this is a duplicated export or a documented repeated record.")
for _, row in files.loc[~files.suffix.eq(".xlsx")].iterrows():
    flag("UNPARSED_ARCHIVE_MEMBER", row.member_path, "Non-XLSX file retained in inventory.",
         "Review format and documentation relevance; excluded from workbook parsing.")
display(files.groupby(["folder", "suffix"]).size().rename("file_count").reset_index())
print("CRC check: passed. Workbook count:", len(xlsx_members))

,folder,suffix,file_count
0,GG_data,.xlsx,46
1,viscosity_data,.xlsx,46


CRC check: passed. Workbook count: 92


## 6. Workbook structure, source headers and column profiles

Each worksheet is inspected independently. Row 1 is treated as the **candidate header row**, consistent with the supplied workbooks. Header anomalies are reported. Profiles include source-column position, exact header, blank cells, numeric coverage, value range, formulas and Excel errors. Records retain original Excel row coordinates.

The audit scans cached values and formula expressions separately. It never evaluates Excel formulas. Numeric statistics describe the readable finite values; their denominator and missing/non-numeric counts remain visible.

<details>
<summary><strong>Data-science rationale</strong></summary>

Schema profiling measures what the source actually contains before a canonical schema is designed. Column positions are retained because duplicate or blank headers can make name-only references ambiguous. A numeric conversion here is diagnostic: it identifies parseable values without changing the raw file or turning an identifier into a predictor.

Missingness includes empty cells and whitespace; textual tokens such as “NA” are not silently converted to missing. Within-sheet duplicate rows are flagged with source coordinates. Identical repeated observations may be legitimate or duplicated acquisition/export records and require contextual review.

A measurement point is nested in a worksheet and an experiment. The number of rows is therefore not an independent sample size. Later validation must account for repeated measurements, sample grouping and possible batch effects to avoid pseudoreplication and information leakage.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Storage modulus G′ and loss modulus G″ describe elastic energy storage and viscous dissipation in oscillatory loading. Their interpretation depends on strain amplitude, frequency, temperature and the material’s state. Linear-viscoelastic interpretation requires evidence that the measurement lies in the linear regime; the presence of modulus columns alone does not establish this.

Steady-shear viscosity η and complex viscosity |η*| are different observables. Complex viscosity depends on oscillation frequency; treating it as interchangeable with steady-shear viscosity requires a justified constitutive or empirical relationship. Shear thinning may support extrusion but does not, by itself, demonstrate shape retention after deposition.

Units are part of the schema. Strain expressed as a fraction and strain expressed as a percentage differ by a factor of 100. The source’s `mPas` viscosity label must be documented before standardization; if confirmed as mPa·s, conversion to Pa·s uses a factor of 0.001. Frequency in Hz and angular frequency in rad/s satisfy ω = 2πf. This audit preserves the reported units and performs none of these conversions.

</details>

In [12]:
def is_blank(value):
    return value is None or (isinstance(value, str) and not value.strip())

def text_value(value):
    return "" if value is None else str(value)

sheet_rows, column_rows, axis_rows = [], [], []
for member in xlsx_members:
    with zipfile.ZipFile(zip_path) as archive:
        payload = archive.read(member)
    try:
        wb_values = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=True)
        wb_formulas = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=False)
    except Exception as exc:
        flag("WORKBOOK_READ_ERROR", member, f"{type(exc).__name__}: {exc}",
             "Inspect this workbook; audit coverage is incomplete.", severity="error")
        continue
    try:
        for ws in wb_values:
            rows = list(ws.iter_rows(values_only=True))
            formula_cells = list(wb_formulas[ws.title].iter_rows())
            if not rows:
                flag("EMPTY_SHEET", member, "No readable cells.", "Review sheet purpose.", sheet=ws.title)
                continue
            width = max(len(r) for r in rows)
            headers = [text_value(v) for v in rows[0]]
            headers += [""] * (width - len(headers))
            data = [(i, list(r) + [None] * (width - len(r))) for i, r in enumerate(rows[1:], 2)]
            active = [(i, r) for i, r in data if not all(is_blank(v) for v in r)]
            blank_rows = [i for i, r in data if all(is_blank(v) for v in r)]
            formulas = [c.coordinate for row in formula_cells for c in row if c.data_type == "f"]
            excel_errors = [c.coordinate for row in formula_cells for c in row if c.data_type == "e"]
            header_problems = any(not h.strip() for h in headers) or len(set(headers)) != len(headers)
            if header_problems:
                flag("HEADER_REVIEW", member, "Blank or duplicate candidate row-1 headers.",
                     "Confirm header row and use column positions before schema mapping.", sheet=ws.title)
            for issue_code, values, action in [
                ("FORMULA_CELLS", formulas, "Check cached results; formulas are not evaluated by this reader."),
                ("EXCEL_ERROR_CELLS", excel_errors, "Inspect original cells and instrument export."),
                ("BLANK_DATA_ROWS", blank_rows, "Check whether blanks separate acquisition segments.")]:
                if values:
                    flag(issue_code, member, json.dumps(values), action, sheet=ws.title)
            seen, duplicate_rows = {}, []
            for i, row in active:
                signature = tuple((type(v).__name__, str(v)) for v in row)
                if signature in seen:
                    duplicate_rows.append(i)
                else:
                    seen[signature] = i
            if duplicate_rows:
                flag("DUPLICATE_MEASUREMENT_ROWS", member, json.dumps(duplicate_rows),
                     "Review acquisition/export context; do not automatically delete.", sheet=ws.title)
            sheet_rows.append(dict(member_path=member, sheet=ws.title, reported_max_row=ws.max_row,
                reported_max_column=ws.max_column, candidate_header_row=1, data_rows=len(active),
                blank_data_rows=len(blank_rows), duplicate_rows=len(duplicate_rows),
                formula_cells=len(formulas), excel_error_cells=len(excel_errors),
                headers_json=json.dumps(headers, ensure_ascii=False)))
            for j, header in enumerate(headers):
                vals = [r[j] for _, r in active]
                blank = [is_blank(v) for v in vals]
                numeric = pd.to_numeric(pd.Series([None if b else v for v, b in zip(vals, blank)], dtype=object), errors="coerce")
                finite = numeric.notna() & np.isfinite(numeric.astype(float))
                clean = numeric[finite].astype(float)
                position = openpyxl.utils.get_column_letter(j + 1)
                col = dict(member_path=member, sheet=ws.title, column_position=j+1, excel_column=position,
                    source_header=header, observed_rows=len(vals), blank_count=sum(blank),
                    blank_percent=100*sum(blank)/len(vals) if vals else 0,
                    numeric_count=int(finite.sum()),
                    nonnumeric_nonblank_count=sum(not b and pd.isna(n) for b,n in zip(blank,numeric)),
                    nonfinite_numeric_count=int((numeric.notna() & ~finite).sum()),
                    distinct_nonblank=len({text_value(v) for v,b in zip(vals,blank) if not b}),
                    numeric_min=clean.min() if len(clean) else None,
                    numeric_max=clean.max() if len(clean) else None,
                    numeric_median=clean.median() if len(clean) else None)
                column_rows.append(col)
                if col['blank_count']:
                    flag("BLANK_MEASUREMENT_CELLS", member, f"{sum(blank)} of {len(vals)} cells blank; column {position}.",
                         "Review missingness before extraction or imputation.", sheet=ws.title, column=header)
                if " in " in header and (col['nonnumeric_nonblank_count'] or col['nonfinite_numeric_count']):
                    flag("NUMERIC_FIELD_REVIEW", member, f"Non-numeric: {col['nonnumeric_nonblank_count']}; nonfinite: {col['nonfinite_numeric_count']}.",
                         "Review source cell values and missing-value conventions.", sheet=ws.title, column=header)
                if header in ["G' in Pa", 'G" in Pa', 'η in mPas', '|η*| in mPas']:
                    affected = [i for (i,_),v in zip(active,numeric) if pd.notna(v) and np.isfinite(v) and v <= 0]
                    if affected:
                        flag("NONPOSITIVE_RHEOLOGY_VALUE", member, "Excel rows: " + json.dumps(affected),
                             "Review measurement range and instrument limits; retain values pending review.", sheet=ws.title, column=header)
                if header in ['ɣ in -', 'ɣ in %', 'f in Hz', 'ɣ̇ in 1/s', 'T in °C']:
                    axis_rows.append({**{k:col[k] for k in ['member_path','sheet','source_header','numeric_min','numeric_max']},
                                      'distinct_numeric_values': int(clean.nunique())})
    finally:
        wb_values.close()
        wb_formulas.close()
workbook_inventory = pd.DataFrame(sheet_rows)
column_profile = pd.DataFrame(column_rows)
axis_profile = pd.DataFrame(axis_rows)
if workbook_inventory.empty:
    raise RuntimeError("No worksheets could be audited; inspect workbook read errors.")
header_inventory = (column_profile.groupby('source_header', dropna=False)
                    .agg(occurrences=('source_header','size'), blank_cells=('blank_count','sum'))
                    .reset_index())
display(header_inventory)
print("Audited worksheets:", len(workbook_inventory))
print("Nonblank data rows across worksheets:", int(workbook_inventory.data_rows.sum()))

,source_header,occurrences,blank_cells
0,"G"" in Pa",46,0
1,G' in Pa,46,0
2,SegIndex,92,0
3,T in °C,92,0
4,f in Hz,46,0
5,t in s,92,0
6,t_seg in s,92,0
7,|η*| in mPas,46,0
8,ɣ in %,2,0
9,ɣ in -,44,0


Audited worksheets: 92
Nonblank data rows across worksheets: 4600


## 7. Unit variants and acquisition-axis evidence

The unit review detects the two observed strain-header forms explicitly. Acquisition-axis summaries show how many distinct numeric values each variable contains within a worksheet. Temperature variation is reported without asserting a tolerance or acceptance limit.

<details>
<summary><strong>Data-science rationale</strong></summary>

Schema drift can occur inside one archive: most files can share a schema while a minority uses a different representation. An exact header inventory makes this visible. A unit difference should become a documented transformation rule, with source values retained, rather than an undocumented numeric adjustment.

Floating-point temperature values may differ slightly across records. Counting distinct values is descriptive, not a test of meaningful thermal instability. Protocol-specific tolerances and measurement uncertainty must come from appropriate documentation.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

A varying strain amplitude at fixed frequency is consistent with an amplitude-dependent test, but the protocol and control mode still require confirmation. A varying shear-rate column provides evidence for a flow-curve acquisition. Neither pattern proves equilibrium, absence of wall slip, adequate instrument sensitivity or independent replication.

For a crosslinked network, a plateau in G′ under an appropriate small-amplitude protocol may characterize stiffness. However, solvent-rich polymer solutions, transient associations and permanent networks can have different responses. Crosslink density cannot be inferred from a filename or directly equated with a single modulus without an appropriate model and measurement conditions.

</details>

In [14]:
strain_headers = set(column_profile.source_header) & {"ɣ in -", "ɣ in %"}
if len(strain_headers) > 1:
    flag("MIXED_STRAIN_UNITS", "rheology archive", "Both fraction and percentage strain headers occur.",
         "Preserve source strain units; define and validate conversion during schema standardization.")
strain_rows = column_profile.loc[column_profile.source_header.isin(strain_headers),
                               ["member_path", "sheet", "source_header", "numeric_min", "numeric_max"]]
display(strain_rows.groupby("source_header").size().rename("column_count").reset_index())
display(strain_rows.loc[strain_rows.source_header.eq("ɣ in %")])
display(axis_profile.head(12))

,source_header,column_count
0,ɣ in %,2
1,ɣ in -,44


,member_path,sheet,source_header,numeric_min,numeric_max
317,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,ɣ in %,0.011359,22.229651
367,GG_data/ALG-I1G-Ph_5.0_HA-HQ-Ph_2.0_22.5C_1.xlsx,ALG_I1G_Ph_5____Ph_2_0_22_5C_1,ɣ in %,0.005846,31.597120


,member_path,sheet,source_header,numeric_min,numeric_max,distinct_numeric_values
0,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,ɣ in -,0.009738,335.218689,50
1,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,f in Hz,1.000000,1.000000,1
2,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,T in °C,22.499475,22.500970,33
3,GG_data/ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,ɣ in -,0.006915,137.641800,50
4,GG_data/ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,f in Hz,1.000000,1.000000,1
5,GG_data/ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,T in °C,22.498285,22.500604,32
6,GG_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0....,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,ɣ in -,0.004146,87.278450,50
7,GG_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0....,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,f in Hz,1.000000,1.000000,1
8,GG_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0....,ALG_I1G_Ph_0___Ph_0_75_22_5C_0,T in °C,22.499017,22.501428,36
9,GG_data/ALG-I1G-Ph_1.0_22.5C_1.xlsx,ALG_I1G_Ph_1_0_22_5C_1_rwd,ɣ in -,0.010928,273.151611,50


## 8. Image-index schema and missingness

The CSV is read as text with blank fields preserved. Column names, row widths, repeated records and candidate image identifiers are inspected. Dates and irradiation-intensity values receive diagnostic parsing in separate objects; raw text remains unchanged.

<details>
<summary><strong>Data-science rationale</strong></summary>

Reading identifiers as strings preserves leading zeros and avoids accidental numeric semantics. Missingness is assessed before conversion: an empty value is not automatically zero, “no” or a negative outcome. Complete-case filtering could preferentially exclude dates or formulations and distort the dataset.

Key uniqueness is a necessary structural check for a proposed image key, but it does not establish that each image represents a different physical construct. A date is a batch-context variable, not a unique experiment identifier. Blank counts, value vocabularies and row-level findings support explicit missing-value and categorical rules in the next phase.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

The source record describes irradiation intensity and SPS presence in the image index. It notes that `completeness_sam` is incomplete and was unused in the reported study. Here, these columns are profiled as observed annotations.

Photochemical processing can alter network formation, but irradiation time, light spectrum, intensity units, formulation chemistry and reagent concentrations are needed to interpret exposure and crosslinking. A numeric intensity value alone cannot establish delivered energy or crosslink density. Blank SPS entries do not establish reagent absence.

Construct morphology can depend on extrusion settings, substrate interactions, recovery, curing and blueprint geometry. Image labels are therefore not substitutes for rheological measurements or biological endpoints. Missing completion labels cannot be converted into successful-print labels.

</details>

In [16]:
with index_path.open('r', encoding='utf-8-sig', newline='') as stream:
    csv_rows = list(csv.reader(stream))
if not csv_rows:
    raise ValueError("The image index is empty.")
headers = csv_rows[0]
if any(not h.strip() for h in headers) or len(set(headers)) != len(headers):
    raise ValueError("CSV has blank or duplicate headers; review before named-column profiling.")
# CSV record numbers are not guaranteed to equal physical lines if fields contain newlines.
malformed = [i for i,r in enumerate(csv_rows[1:], 2) if len(r) != len(headers)]
if malformed:
    raise ValueError("CSV record widths differ from the header: " + repr(malformed[:20]))
image_index = pd.DataFrame(csv_rows[1:], columns=headers)
expected_columns = ['image_id','date','ink_type_ph','shape','intensity','completeness_sam','contain_sps']
if headers != expected_columns:
    flag("IMAGE_SCHEMA_DIFFERENCE", index_path.name, json.dumps(headers),
         "Review header order, added fields and missing expected columns.")
missing_mask = image_index.apply(lambda s: s.str.strip().eq(''))
image_profile = pd.DataFrame({
    'column': headers, 'rows': len(image_index),
    'blank_count': [int(missing_mask[c].sum()) for c in headers],
    'blank_percent': [float(missing_mask[c].mean()*100) for c in headers],
    'distinct_nonblank': [int(image_index.loc[~missing_mask[c],c].nunique()) for c in headers],
})
for row in image_profile.itertuples():
    if row.blank_count:
        flag("IMAGE_METADATA_BLANK", index_path.name, f"{row.blank_count} of {row.rows} records blank ({row.blank_percent:.1f}%).",
             "Retain as unknown; investigate collection/annotation coverage.", column=row.column)

def record_numbers(mask):
    return (image_index.index[mask] + 2).tolist()

dup_rows = image_index.duplicated(keep=False)
if dup_rows.any():
    flag("DUPLICATE_IMAGE_RECORD", index_path.name, json.dumps(record_numbers(dup_rows)),
         "Check whether records were duplicated during index assembly.")
if 'image_id' in headers:
    duplicate_id = image_index.image_id.duplicated(keep=False) & ~missing_mask.image_id
    if duplicate_id.any():
        flag("NONUNIQUE_IMAGE_ID", index_path.name, json.dumps(record_numbers(duplicate_id)),
             "Resolve identifier granularity before joins.", column='image_id')
if 'date' in headers:
    parsed_dates = pd.to_datetime(image_index.date, format='%Y/%m/%d', errors='coerce')
    bad_dates = parsed_dates.isna() & ~missing_mask.date
    if bad_dates.any():
        flag("DATE_PARSE_REVIEW", index_path.name, json.dumps(record_numbers(bad_dates)),
             "Check source date convention; preserve raw text.", column='date')
if 'intensity' in headers:
    intensity_numeric = pd.to_numeric(image_index.intensity, errors='coerce')
    bad_intensity = (~np.isfinite(intensity_numeric) | (intensity_numeric < 0)) & ~missing_mask.intensity
    if bad_intensity.any():
        flag("INTENSITY_VALUE_REVIEW", index_path.name, json.dumps(record_numbers(bad_intensity)),
             "Confirm value conventions and units.", column='intensity')
    flag("INTENSITY_UNIT_UNRESOLVED", index_path.name, "CSV header does not specify intensity units.",
         "Check experimental methods before conversion or dose calculations.", column='intensity')
frequency_rows = []
for c in headers:
    if c == 'image_id':
        continue
    for value, count in image_index[c].value_counts(dropna=False).items():
        frequency_rows.append(dict(column=c, source_value=value, is_blank=not value.strip(), count=int(count)))
image_value_counts = pd.DataFrame(frequency_rows)
display(image_profile)
display(image_value_counts.loc[image_value_counts.column.isin(['shape','contain_sps','completeness_sam','intensity'])])
print("Image-index records:", len(image_index))

,column,rows,blank_count,blank_percent,distinct_nonblank
0,image_id,1031,0,0.000000,1031
1,date,1031,0,0.000000,18
2,ink_type_ph,1031,0,0.000000,48
3,shape,1031,0,0.000000,4
4,intensity,1031,0,0.000000,7
5,completeness_sam,1031,943,91.464597,1
6,contain_sps,1031,922,89.427740,1


,column,source_value,is_blank,count
66,shape,grid,False,766
67,shape,star,False,106
68,shape,nose,False,90
69,shape,ichou,False,69
70,intensity,0,False,261
71,intensity,100,False,216
72,intensity,50,False,197
73,intensity,1,False,148
74,intensity,25,False,111
75,intensity,75,False,94


Image-index records: 1031


## 9. Candidate filename correspondences and linkage boundaries

Exact workbook stems are compared across the two observed source folders. The resulting table is an inventory of candidate correspondences, not a merged experimental dataset. Image identifiers are listed as expected names according to the source naming convention; image existence remains untested.

<details>
<summary><strong>Data-science rationale</strong></summary>

Record linkage needs a defined entity and a reliable key. A shared filename stem can support a candidate correspondence between two exports, but cannot establish one-to-one physical sample identity. The comparison retains both member paths and match cardinalities, preventing an accidental many-to-many join from inflating observations.

The image-index formulation labels and workbook labels use different conventions. No fuzzy matching or inferred formulation mapping is applied. A future crosswalk should state its source evidence, ambiguity and expected cardinality. Train/test grouping cannot be selected responsibly until these relationships are established.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

Two tests carrying the same formulation label may have used the same sample, fresh aliquots, independent preparations or different batches. Oscillatory loading can also alter a sample before a subsequent test. These possibilities affect whether rheological responses can be compared or pooled.

A numeric suffix is retained as a source token; it is not promoted to a replicate identifier. Formulation, batch, physical sample, test run, printed construct and image are distinct material entities. Their relationships must be supported by documentation before structure–process–property comparisons or prediction.

</details>

In [18]:
member_groups = {}
for member in xlsx_members:
    p = PurePosixPath(member)
    member_groups.setdefault(p.stem, {}).setdefault(p.parent.as_posix(), []).append(member)
link_rows = []
for stem, groups in sorted(member_groups.items()):
    gg = groups.get('GG_data', [])
    vis = groups.get('viscosity_data', [])
    link_rows.append(dict(source_stem=stem, GG_count=len(gg), viscosity_count=len(vis),
        GG_members=json.dumps(gg), viscosity_members=json.dumps(vis),
        exact_one_each=len(gg)==1 and len(vis)==1, physical_sample_link='unresolved'))
filename_correspondences = pd.DataFrame(link_rows)
flag("PHYSICAL_SAMPLE_LINK_UNRESOLVED", "cross-source linkage",
     "Filename correspondences are not verified sample or replicate identifiers.",
     "Obtain a source-supported formulation/batch/sample/run crosswalk before merging.")
flag("IMAGE_FILES_NOT_AUDITED", "images.zip", "Image archive is outside the selected inputs.",
     "Check indexed filenames against the image archive in a later audit.")
if 'image_id' in image_index.columns:
    image_file_expectations = pd.DataFrame({
        'source_csv_record': image_index.index+2,
        'image_id': image_index.image_id,
        'expected_filename': image_index.image_id.map(lambda v: f'resized_IMG_{v}.png' if v.strip() else ''),
        'existence_status': 'not tested: image archive not supplied',
    })
else:
    image_file_expectations = pd.DataFrame(columns=['source_csv_record','image_id','expected_filename','existence_status'])
display(filename_correspondences.head(8))
print("Candidate stems:", len(filename_correspondences))
print("Stems with exactly one workbook in each observed folder:", int(filename_correspondences.exact_one_each.sum()))

,source_stem,GG_count,viscosity_count,GG_members,viscosity_members,exact_one_each,physical_sample_link
0,ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0,1,1,"[""GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0...","[""viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_...",True,unresolved
1,ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_22.5C_0,1,1,"[""GG_data/ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_22.5C_0...","[""viscosity_data/ALG-I1G-Ph_0.5_HA-HQ-Ph_0.75_...",True,unresolved
2,ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0,1,1,"[""GG_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_...","[""viscosity_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75...",True,unresolved
3,ALG-I1G-Ph_1.0_22.5C_0,0,1,[],"[""viscosity_data/ALG-I1G-Ph_1.0_22.5C_0.xlsx""]",False,unresolved
4,ALG-I1G-Ph_1.0_22.5C_1,1,0,"[""GG_data/ALG-I1G-Ph_1.0_22.5C_1.xlsx""]",[],False,unresolved
5,ALG-I1G-Ph_1.0_HA-HQ-Ph_0.25_22.5C_1,1,1,"[""GG_data/ALG-I1G-Ph_1.0_HA-HQ-Ph_0.25_22.5C_1...","[""viscosity_data/ALG-I1G-Ph_1.0_HA-HQ-Ph_0.25_...",True,unresolved
6,ALG-I1G-Ph_1.0_HA-HQ-Ph_0.5_22.5C_0,1,1,"[""GG_data/ALG-I1G-Ph_1.0_HA-HQ-Ph_0.5_22.5C_0....","[""viscosity_data/ALG-I1G-Ph_1.0_HA-HQ-Ph_0.5_2...",True,unresolved
7,ALG-I1G-Ph_1.0_HA-HQ-Ph_1.0_22.5C_0,1,0,"[""GG_data/ALG-I1G-Ph_1.0_HA-HQ-Ph_1.0_22.5C_0....",[],False,unresolved


Candidate stems: 62
Stems with exactly one workbook in each observed folder: 30


## 10. Audit findings and reproducible exports

Audit tables are written to a dedicated metadata folder. These exports provide the evidence needed for Phase 2 schema design. Timestamp and software versions are stored in a run manifest; source hashes are rechecked before writing to confirm input preservation.

<details>
<summary><strong>Data-science rationale</strong></summary>

Idempotence means a repeated run reconstructs the same substantive tables from unchanged inputs. Named outputs are replaced rather than appended, preventing duplicated audit records. The run timestamp changes by design; source hashes and audit tables should agree across runs with the same inputs and dependencies.

The issue log is a review queue. Structural errors, missing context and potentially unusual physical values have different implications. A successful run means the code completed its declared checks; unresolved findings remain visible and do not become “passed” merely because a file was exported.

</details>

<details>
<summary><strong>Materials-science context</strong></summary>

The handoff should preserve units, measurement axes and links to original workbooks. Schema design can then represent formulations, measurement runs and repeated points separately, with explicit source-unit fields and missing-value rules.

Quantitative hydrogel comparison will require comparable test conditions and reliable experimental grouping. Statistical models should follow a defensible definition of the prediction target, independent observational unit and validation split. This audit provides evidence for those choices but does not fit constitutive models, estimate crosslink density or claim printability from the source labels.

</details>

In [20]:
issue_columns = ['issue_code','severity','source','sheet','column','evidence','next_action']
issue_log = pd.DataFrame(issues, columns=issue_columns).sort_values(
    ['issue_code','source','sheet','column'], kind='stable').reset_index(drop=True)
issue_log.insert(0,'issue_id',[f'AUDIT-{i:04d}' for i in range(1,len(issue_log)+1)])
for path, row in zip((zip_path,index_path), source_manifest.itertuples()):
    if digest(path,'sha256') != row.sha256:
        raise RuntimeError("An input changed during the audit; outputs were not written.")
exports = {
    'source_manifest.csv': source_manifest,
    'archive_member_inventory.csv': archive_inventory,
    'workbook_structure_inventory.csv': workbook_inventory,
    'source_header_inventory.csv': header_inventory,
    'measurement_column_profile.csv': column_profile,
    'measurement_axis_profile.csv': axis_profile,
    'image_index_column_profile.csv': image_profile,
    'image_index_value_counts.csv': image_value_counts,
    'candidate_filename_correspondences.csv': filename_correspondences,
    'image_filename_expectations.csv': image_file_expectations,
    'audit_issue_log.csv': issue_log,
}
output_dir.mkdir(parents=True, exist_ok=True)
for name, frame in exports.items():
    frame.to_csv(output_dir/name, index=False, encoding='utf-8-sig', lineterminator='\n')
summary = {
    'run_utc': RUN_UTC, 'source_record': 'https://zenodo.org/records/19602891',
    'source_metadata_review_date': '2026-10-02',
    'environment': environment.to_dict(orient='records'),
    'xlsx_files_in_archive': len(xlsx_members),
    'xlsx_files_audited': int(workbook_inventory.member_path.nunique()),
    'worksheets_audited': len(workbook_inventory),
    'measurement_rows': int(workbook_inventory.data_rows.sum()),
    'image_index_rows': len(image_index), 'issues': len(issue_log),
    'input_sha256': dict(zip(source_manifest.canonical_filename,source_manifest.sha256)),
    'export_sha256': {name: digest(output_dir/name,'sha256') for name in exports},
    'dataset_licence': 'unresolved', 'sample_linkage': 'unresolved',
    'image_existence': 'not tested', 'raw_files_preserved': True,
}
(output_dir/'audit_run_manifest.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
display(issue_log.groupby(['severity','issue_code']).size().rename('finding_count').reset_index())
print('Audit exports:', len(exports), 'CSV files and 1 JSON run manifest.')
print('Output location:', output_dir.relative_to(project_root).as_posix())
print('Original input hashes unchanged: yes')

,severity,issue_code,finding_count
0,review,DATASET_LICENCE_UNRESOLVED,1
1,review,IMAGE_FILES_NOT_AUDITED,1
2,review,IMAGE_METADATA_BLANK,2
3,review,INTENSITY_UNIT_UNRESOLVED,1
4,review,MIXED_STRAIN_UNITS,1
5,review,NONPOSITIVE_RHEOLOGY_VALUE,8
6,review,PHYSICAL_SAMPLE_LINK_UNRESOLVED,1


Audit exports: 11 CSV files and 1 JSON run manifest.
Output location: data/metadata/zenodo_source_audit
Original input hashes unchanged: yes


## 11. Checkpoint interpretation and next phase

The tables above establish source-file integrity, workbook and column structure, observed unit variants, missing metadata and candidate filename correspondences. The recorded row count is a count of measurement points across sheets, **not a count of independent formulations or samples**.

### Phase 2 handoff

- A canonical measurement schema needs fields for original member path, sheet, Excel row, variable, source value, source unit and eventual standardized representation.
- Strain representation needs explicit fraction/percentage handling. Frequency and shear rate remain distinct physical axes.
- Experimental identifiers and replicate relationships remain unresolved until supported by documentation.
- Missing image-index annotations remain unknown. No blank completion or reagent field becomes an assumed outcome.
- Dataset rights, intensity units, preparation details, instrument conditions and image existence remain review items where unresolved.

<details>
<summary><strong>Reopening and rerunning</strong></summary>

The expected notebook location is `notebooks/01_zenodo_source_audit.ipynb`. The two inputs remain in `data/raw/zenodo_19602891/`. A fresh-kernel **Run All** reruns imports, path selection, source checks, profiles and exports in order. No variables from Stage 00 are required. The generated audit files are rebuilt rather than appended. Input selection and errors are reported explicitly.

</details>

### References and provenance

- [Zenodo record and download metadata](https://zenodo.org/records/19602891) — reference version and checksums reviewed on 2 October 2026; local file observations are generated by this notebook.
- [Associated publication](https://doi.org/10.1080/17452759.2026.2671497) — methodological reference for further review, not a claim that all experimental details have been independently verified here.
- [Authors’ code repository](https://github.com/KORINZ/generative-ai-bioprinting-framework) — reference for subsequent interpretation of source workflows.

The fixed metadata review date and the current run timestamp have different meanings. The former records when reference metadata were checked; the latter records when the local audit was executed.